# QML Intrusion Detection — IoT-23 Dataset
> Stratosphere Lab / CTU — `CTU-IoT-Malware-Capture-*` captures
>
> Implementation of: *Kalinin & Krundyshev (2023)*

---
**Your dataset structure should look like:**
```
your_folder/
├── CTU-IoT-Malware-Capture-1-1/
│   └── conn.log.labeled      ← or inside bro/ subfolder
├── CTU-IoT-Malware-Capture-3-1/
│   └── conn.log.labeled
└── ...
```

## 0. Configuration — Set your dataset path here

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# ── SET THIS to the folder containing your CTU-IoT-Malware-Capture-* folders ──
DATASET_DIR = r'C:\Users\YourName\Downloads\IoT-23'   # Windows example
# DATASET_DIR = '/home/user/datasets/IoT-23'           # Linux/Mac example

N_SAMPLES    = 2000   # samples to use for QML training (increase for better accuracy)
N_QUBITS     = 4      # qubits for QSVM/QCNN
RANDOM_STATE = 42
# ──────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import os
os.makedirs('results', exist_ok=True)

from data.iot23_loader import IoT23Loader
from encoder.qubit_encoder import QuantumEncoder
from models.qsvm_model import QSVMDetector, SVMBaseline
from models.qcnn_model import QCNNDetector
from utils.evaluation import Evaluator

print('Setup complete.')

## 1. Load IoT-23 Dataset

In [ ]:
loader = IoT23Loader(
    dataset_dir=DATASET_DIR,
    binary=False,              # 5-class: Benign / DDoS / C&C / PortScan / Botnet / ...
    max_rows_per_capture=50_000
)

print('Capture folders found:')
for cap in loader.list_captures():
    print(' ', cap)

In [ ]:
# Load all captures
X_all, y_all = loader.load_all()
print(f'\nTotal: {X_all.shape[0]:,} flows × {X_all.shape[1]} features')

In [ ]:
# Class distribution
classes, counts = np.unique(y_all, return_counts=True)

fig, ax = plt.subplots(figsize=(10, 4))
colors = plt.cm.Set2(np.linspace(0, 1, len(classes)))
bars = ax.bar(classes, counts, color=colors, edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{count:,}', ha='center', va='bottom', fontsize=9)
ax.set_title('IoT-23 Dataset — Traffic Label Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Flow count')
ax.set_xlabel('Label')
plt.xticks(rotation=20, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/iot23_class_distribution.png', dpi=150)
plt.show()

## 2. Prepare Training / Test Split

In [ ]:
# Stratified sample for QML (simulator can't handle millions of flows)
from sklearn.model_selection import train_test_split

# Remove classes with too few samples for stratification
class_counts = dict(zip(*np.unique(y_all, return_counts=True)))
valid_mask   = np.array([class_counts[y] >= 10 for y in y_all])
X_valid, y_valid = X_all[valid_mask], y_all[valid_mask]

# Sample N_SAMPLES rows
if N_SAMPLES < len(X_valid):
    _, X_s, _, y_s = train_test_split(
        X_valid, y_valid,
        test_size=N_SAMPLES / len(X_valid),
        stratify=y_valid,
        random_state=RANDOM_STATE
    )
else:
    X_s, y_s = X_valid, y_valid

X_train, X_test, y_train, y_test = train_test_split(
    X_s, y_s, test_size=0.2, stratify=y_s, random_state=RANDOM_STATE
)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Classes: {sorted(set(y_train))}')

## 3. Quantum Encoding Demo

In [ ]:
import cirq
from encoder.qubit_encoder import QuantumEncoder

# Features are already scaled to [0, π] by the loader
enc = QuantumEncoder(backend='cirq', fit_scaler=False)

print('Encoding first 3 flows as Cirq circuits...')
circuits = enc.encode_batch(X_train[:3], verbose=True)

print(f'\nSample circuit for flow 0 (first 6 Rx gates):')
q = cirq.GridQubit(1, 1)
demo = cirq.Circuit()
for val in X_train[0, :6]:
    demo.append(cirq.rx(float(val))(q))
print(demo)

## 4. Train Models

### 4a. Classical SVM (Baseline)

In [ ]:
svm = SVMBaseline(kernel='rbf', C=1.0)
svm.fit(X_train, y_train)

print('\n=== Classical SVM — Classification Report ===')
print(svm.report(X_test, y_test))

### 4b. QSVM (Quantum Kernel)

In [ ]:
qsvm = QSVMDetector(backend='cirq_sim', n_qubits=N_QUBITS, C=1.0)
qsvm.fit(X_train, y_train)

print('\n=== QSVM — Classification Report ===')
print(qsvm.report(X_test, y_test))

## 5. Evaluation — Confusion Matrices & ROC Curves

In [ ]:
CLASS_NAMES = sorted(set(y_train))
ev = Evaluator(class_names=CLASS_NAMES, save_dir='results')

# --- SVM ---
y_true_svm  = svm.le.transform(y_test)
y_pred_svm  = svm.predict(X_test)

try:
    y_prob_svm = svm.model.predict_proba(svm._preprocess(X_test))
    ev.plot_combined(y_true_svm, y_pred_svm, y_prob_svm, title='SVM (IoT-23)')
except Exception:
    ev.plot_confusion(y_true_svm, y_pred_svm, title='SVM (IoT-23)')

# --- QSVM ---
y_true_qsvm = qsvm.le.transform(y_test)
y_pred_qsvm = qsvm.predict(X_test)

try:
    y_prob_qsvm = qsvm.predict_proba(X_test)
    ev.plot_combined(y_true_qsvm, y_pred_qsvm, y_prob_qsvm, title='QSVM (IoT-23)')
except Exception:
    ev.plot_confusion(y_true_qsvm, y_pred_qsvm, title='QSVM (IoT-23)')

print('Plots saved to results/')

## 6. Accuracy Comparison

In [ ]:
acc_svm  = accuracy_score(y_true_svm,  y_pred_svm)
acc_qsvm = accuracy_score(y_true_qsvm, y_pred_qsvm)

fig, ax = plt.subplots(figsize=(7, 4))
models  = ['SVM (Classical)', 'QSVM (Quantum)']
accs    = [acc_svm * 100, acc_qsvm * 100]
colors  = ['#4C72B0', '#DD8452']

bars = ax.bar(models, accs, color=colors, edgecolor='white', width=0.4, linewidth=1.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            f'{acc:.1f}%', ha='center', fontsize=12, fontweight='bold')

ax.axhline(98, color='green', linestyle='--', alpha=0.6, label='Paper target (98%)')
ax.set_ylim(0, 110)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('SVM vs QSVM — IoT-23 Dataset', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/accuracy_comparison_iot23.png', dpi=150)
plt.show()

print(f'SVM  accuracy : {acc_svm*100:.2f}%')
print(f'QSVM accuracy : {acc_qsvm*100:.2f}%')
print(f'Improvement   : +{(acc_qsvm - acc_svm)*100:.2f}%')

## 7. Training Time Benchmark (Table 4)
Reproduces the paper's Table 4 for this dataset.

In [ ]:
import time, copy

SIZES = [50, 100, 200, 500, 1000]
timing = {'SVM': [], 'QSVM': []}

for n in SIZES:
    idx = np.random.choice(len(X_s), min(n, len(X_s)), replace=False)
    Xn, yn = X_s[idx], y_s[idx]

    for label, ModelClass, kwargs in [
        ('SVM',  SVMBaseline,  {}),
        ('QSVM', QSVMDetector, {'backend': 'cirq_sim', 'n_qubits': N_QUBITS}),
    ]:
        m  = ModelClass(**kwargs)
        t0 = time.time()
        m.fit(Xn, yn)
        elapsed = time.time() - t0
        timing[label].append(elapsed)
        print(f'  {label:5s}  n={n:>5,}  {elapsed:.2f}s')

fig, ax = plt.subplots(figsize=(8, 5))
for label, times in timing.items():
    ax.plot(SIZES, times, marker='o', linewidth=2, label=label)
ax.set_xlabel('Training set size', fontsize=12)
ax.set_ylabel('Time (s)', fontsize=12)
ax.set_title('Training Time: SVM vs QSVM — IoT-23', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/training_time_iot23.png', dpi=150)
plt.show()

## 8. (Optional) QCNN
Uncomment to run QCNN — slower but achieves better accuracy per the paper.

In [ ]:
# qcnn = QCNNDetector(n_qubits=4, n_layers=1, backend='cirq_sim', epochs=10, batch_size=16)
# qcnn.fit(X_train, y_train)
# print(qcnn.report(X_test, y_test))
print('Uncomment above to run QCNN.')